# Open-Source Models — Base Benchmark Generation

This notebook runs open-source language models on the Turkish reasoning base
benchmark.

Models from the original open-source experiment:

- Qwen 2.5 7B Instruct
- Trendyol LLM 7B Chat
- Llama 3.1 8B Instruct
- Gemma 2 9B IT

Models are loaded locally through Hugging Face Transformers using 4-bit
quantization.

For each benchmark question, the notebook stores:

- question identifier,
- model reasoning,
- final answer,
- confidence,
- complete raw model response,
- generation errors, if any.

This notebook handles **generation only**.

Evaluation is performed separately.

## Important safety and reproducibility notes

Model execution is disabled by default.

No model is downloaded or run unless:

```python
RUN_GENERATION = True

## 1. Install dependencies

Open-source generation uses:

- PyTorch
- Hugging Face Transformers
- Accelerate
- bitsandbytes
- SentencePiece
- Hugging Face Hub

The evaluation libraries used by the historical notebook are intentionally not
installed here because evaluation is handled separately.

In [28]:
%pip install -q transformers accelerate bitsandbytes sentencepiece huggingface_hub pandas tqdm

In [29]:
from pathlib import Path
from datetime import datetime, timezone

import gc
import hashlib
import json
import os
import re
import sys

import pandas as pd
from tqdm.auto import tqdm


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

Python: 3.13.15
pandas: 2.2.3


## 2. Locate the project

The notebook supports:

1. a cloned GitHub repository,
2. a project copy stored in Google Drive.

Google Drive is particularly useful for long Colab runs because checkpoints
remain available if the runtime disconnects.

In [30]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

USE_GOOGLE_DRIVE = True

DRIVE_PROJECT_PATH = Path(
    "/content/drive/MyDrive/turkish-llm-reasoning"
)


if USE_GOOGLE_DRIVE:

    try:
        from google.colab import drive

        drive.mount("/content/drive")

        PROJECT_ROOT = DRIVE_PROJECT_PATH

    except ImportError:

        raise RuntimeError(
            "Google Drive mounting is only available in Google Colab."
        )


else:

    current_path = Path.cwd().resolve()

    candidates = [
        current_path,
        *current_path.parents,
    ]

    PROJECT_ROOT = None


    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):

            PROJECT_ROOT = candidate
            break


    if PROJECT_ROOT is None:

        raise FileNotFoundError(
            "Project root could not be located. "
            "Run the notebook from inside the repository "
            "or enable Google Drive mode."
        )


print("Project root:")
print(PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root:
/content/drive/MyDrive/turkish-llm-reasoning


## 3. Dataset configuration and reproducibility

Two base-dataset profiles are supported:

- **original** — the snapshot used in the historical experiments.
- **current** — an optional evolving benchmark for future experiments.

The historical profile performs strict size checks.

The current profile does not assume a fixed number of questions.

A SHA-256 hash identifies the exact dataset file used by each run.

In [31]:
# ============================================================
# DATASET CONFIGURATION
# ============================================================

DATASET_PROFILE = "smoke"


DATASET_PROFILES = {

    "original": {
        "path": (
            PROJECT_ROOT
            / "data"
            / "base"
            / "base_questions.csv"
        ),

        "expected_raw_rows": 489,
        "expected_usable_rows": 488,

        "description": (
            "Original base benchmark snapshot"
        ),
    },


    "current": {
        "path": (
            PROJECT_ROOT
            / "data"
            / "base"
            / "base_questions_current.csv"
        ),

        "expected_raw_rows": None,
        "expected_usable_rows": None,

        "description": (
            "Current evolving base benchmark"
        ),
    },

    "smoke": {
    "path": (
        PROJECT_ROOT
        / "data"
        / "base"
        / "smoke_base_v1.csv"
    ),

    "expected_raw_rows": 5,
    "expected_usable_rows": 5,

    "description": (
        "Small smoke-test dataset for pipeline validation"
    ),
    },
}


if DATASET_PROFILE not in DATASET_PROFILES:

    raise ValueError(
        f"Unknown dataset profile: {DATASET_PROFILE}"
    )


DATASET_CONFIG = DATASET_PROFILES[
    DATASET_PROFILE
]

BASE_DATA_PATH = DATASET_CONFIG[
    "path"
]


print("Dataset profile:")
print(f"  Name:        {DATASET_PROFILE}")
print(f"  Description: {DATASET_CONFIG['description']}")
print(f"  Path:        {BASE_DATA_PATH}")

Dataset profile:
  Name:        smoke
  Description: Small smoke-test dataset for pipeline validation
  Path:        /content/drive/MyDrive/turkish-llm-reasoning/data/base/smoke_base_v1.csv


In [32]:
def calculate_file_sha256(filepath):

    sha256 = hashlib.sha256()

    with open(filepath, "rb") as file:

        for chunk in iter(
            lambda: file.read(8192),
            b"",
        ):

            sha256.update(chunk)

    return sha256.hexdigest()

In [33]:
if not BASE_DATA_PATH.exists():

    raise FileNotFoundError(
        f"Base dataset not found: {BASE_DATA_PATH}"
    )


raw_df = pd.read_csv(
    BASE_DATA_PATH
)


REQUIRED_FIELDS = [
    "question_id",
    "category",
    "difficulty",
    "context",
    "question",
    "gold_evidence",
    "gold_answer",
]


missing_columns = [
    column
    for column in REQUIRED_FIELDS
    if column not in raw_df.columns
]


if missing_columns:

    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


df = (
    raw_df
    .dropna(
        subset=REQUIRED_FIELDS
    )
    .copy()
)


DATASET_SHA256 = calculate_file_sha256(
    BASE_DATA_PATH
)


print(f"Raw dataset:     {len(raw_df)} rows")
print(f"Usable dataset:  {len(df)} rows")
print(f"Excluded:        {len(raw_df) - len(df)} rows")
print(f"Dataset SHA-256: {DATASET_SHA256}")

Raw dataset:     5 rows
Usable dataset:  5 rows
Excluded:        0 rows
Dataset SHA-256: 20d742f0441f593d59cc929c7c8d7b9cc9c1efbd10b8224dbc4519935e8e61f9


In [34]:
expected_raw_rows = DATASET_CONFIG[
    "expected_raw_rows"
]

expected_usable_rows = DATASET_CONFIG[
    "expected_usable_rows"
]


if (
    expected_raw_rows is not None
    and len(raw_df) != expected_raw_rows
):

    raise ValueError(
        f"Dataset profile '{DATASET_PROFILE}' expects "
        f"{expected_raw_rows} raw rows, "
        f"but {len(raw_df)} were found."
    )


if (
    expected_usable_rows is not None
    and len(df) != expected_usable_rows
):

    raise ValueError(
        f"Dataset profile '{DATASET_PROFILE}' expects "
        f"{expected_usable_rows} usable rows, "
        f"but {len(df)} were found."
    )


if not raw_df["question_id"].is_unique:

    raise ValueError(
        "question_id values must be unique."
    )


print("✓ Dataset validation passed.")

✓ Dataset validation passed.


### Historical dataset note

The original base benchmark contained **489 records**.

One record lacked the reference fields required by the experiment pipeline,
leaving **488 usable questions**.

These numbers describe the historical dataset snapshot only.

## 4. Generation configuration and model registry

The registry contains the open-source models used in the original experiment.

New Hugging Face causal language models can be added by creating another
registry entry.

The historical generation configuration used:

- 4-bit quantization,
- NF4 quantization,
- float16 computation,
- double quantization,
- greedy generation (`do_sample=False`),
- `max_new_tokens=768`.

A model may use a system message when supported by its chat template.
Otherwise, the system instructions are combined with the user message.

In [35]:
# ============================================================
# GENERATION CONFIGURATION
# ============================================================

RUN_GENERATION = True

CONFIRM_MULTIPLE_MODELS = False

RUN_LABEL = "smoke_v1"

CHECKPOINT_EVERY = 5


MODEL_REGISTRY = {

    "qwen2.5-7b-instruct": {
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "display_name": "Qwen 2.5 7B Instruct",
        "historical": True,
        "max_new_tokens": 768,
    },

    "trendyol-llm-7b-chat": {
        "model_id": "Trendyol/Trendyol-LLM-7b-chat-v1.0",
        "display_name": "Trendyol LLM 7B Chat",
        "historical": True,
        "max_new_tokens": 768,
    },

    "llama-3.1-8b-instruct": {
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "display_name": "Llama 3.1 8B Instruct",
        "historical": True,
        "max_new_tokens": 768,
    },

    "gemma-2-9b-it": {
        "model_id": "google/gemma-2-9b-it",
        "display_name": "Gemma 2 9B IT",
        "historical": True,
        "max_new_tokens": 768,
    },
}


ORIGINAL_OPEN_MODELS = [
    "qwen2.5-7b-instruct",
    "trendyol-llm-7b-chat",
    "llama-3.1-8b-instruct",
    "gemma-2-9b-it",
]


# Safe default: one model.
SELECTED_MODELS = [
    "qwen2.5-7b-instruct",
]

In [36]:
if not SELECTED_MODELS:

    raise ValueError(
        "At least one model must be selected."
    )


unknown_models = [
    model
    for model in SELECTED_MODELS
    if model not in MODEL_REGISTRY
]


if unknown_models:

    raise ValueError(
        f"Unknown model(s): {unknown_models}"
    )


if len(SELECTED_MODELS) != len(
    set(SELECTED_MODELS)
):

    raise ValueError(
        "SELECTED_MODELS contains duplicate entries."
    )


print("Selected models:")

for model_label in SELECTED_MODELS:

    config = MODEL_REGISTRY[
        model_label
    ]

    print(
        f"  - {config['display_name']} "
        f"({config['model_id']})"
    )

Selected models:
  - Qwen 2.5 7B Instruct (Qwen/Qwen2.5-7B-Instruct)


## 5. GPU and Hugging Face access

Generation requires a CUDA-capable GPU.

A Hugging Face token can be provided through:

- the `HF_TOKEN` environment variable, or
- Google Colab Secrets.

The token is not written to disk or stored in notebook outputs.

Some model repositories may additionally require the user to accept their
license or access conditions on Hugging Face.

In [37]:
def get_secret(name):

    value = os.getenv(name)

    if value:
        return value


    try:

        from google.colab import userdata

        return userdata.get(name)

    except Exception:

        return None

In [38]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


if RUN_GENERATION:

    if not torch.cuda.is_available():

        raise RuntimeError(
            "A CUDA-capable GPU is required for generation."
        )


    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            1,
        ),
        "GB",
    )


else:

    print(
        "Generation is disabled. "
        "GPU availability is not required yet."
    )

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


In [39]:
def login_to_huggingface_if_available():

    token = get_secret(
        "HF_TOKEN"
    )


    if not token:

        print(
            "HF_TOKEN not found. "
            "Continuing without authenticated login."
        )

        return


    from huggingface_hub import login

    login(
        token=token,
        add_to_git_credential=False,
    )

    print(
        "✓ Hugging Face authentication completed."
    )

## 6. Prompt

The system prompt preserves the reasoning / answer / confidence format used in
the original project.

The open-source experiment used the same system instructions as the
closed-model experiment, with an explicit JSON-only reminder in the user
message.

In [40]:
SYSTEM_PROMPT = """Sen bir problem çözme asistanısın. Sana verilen bağlam (context) ve soruyu (question) analiz edeceksin.

GÖREV:
1. Bağlamdaki bilgileri kullanarak adım adım düşün.
2. Her adımda hangi bilgiyi kullandığını açıkça belirt.
3. Sonuca ulaş ve cevabını ver.

ÇIKTI FORMATI (JSON):
{
    "reasoning": "Adım adım düşünce sürecin. Her adımda bağlamdan hangi bilgiyi kullandığını yaz.",
    "answer": "Nihai cevabın (kısa ve net)",
    "confidence": 0-100 arası bir sayı (cevabından ne kadar eminsin)
}

ÖNEMLİ:
- Sadece bağlamda verilen bilgileri kullan, dışarıdan bilgi ekleme.
- Cevabın SADECE geçerli bir JSON objesi olmalı, başka bir şey yazma.
- Türkçe yanıt ver."""

In [41]:
def build_user_prompt(
    context,
    question,
):

    return f"""BAĞLAM:
{context}

SORU:
{question}

Lütfen yukarıdaki kurallara uygun olarak SADECE geçerli JSON formatında cevap ver."""

In [42]:
PROMPT_REFERENCE = (
    SYSTEM_PROMPT
    + "\n---USER-PROMPT---\n"
    + build_user_prompt(
        "{context}",
        "{question}",
    )
)


PROMPT_SHA256 = hashlib.sha256(
    PROMPT_REFERENCE.encode("utf-8")
).hexdigest()


print("Prompt SHA-256:")
print(PROMPT_SHA256)

Prompt SHA-256:
5952958918c52f85e443ce41434418425c3442c85c7d244e1ca3a0f13aa8adaa


## 7. JSON response parser

The parser attempts to recover a JSON object from the generated text.

The complete raw response is always preserved, including when parsing fails.

This is important because parsing logic can be improved later without rerunning
the language model.

In [43]:
def extract_json_from_response(content):

    if content is None:
        return None


    content = str(
        content
    ).strip()


    if not content:
        return None


    try:

        parsed = json.loads(
            content
        )

        if isinstance(parsed, dict):
            return parsed

    except json.JSONDecodeError:
        pass


    code_blocks = re.findall(
        r"```(?:json)?\s*(.*?)```",
        content,
        flags=(
            re.DOTALL
            | re.IGNORECASE
        ),
    )


    for block in code_blocks:

        try:

            parsed = json.loads(
                block.strip()
            )

            if isinstance(parsed, dict):
                return parsed

        except json.JSONDecodeError:
            continue


    start = content.find("{")
    end = content.rfind("}")


    if (
        start != -1
        and end > start
    ):

        candidate = content[
            start:end + 1
        ]


        try:

            parsed = json.loads(
                candidate
            )

            if isinstance(parsed, dict):
                return parsed

        except json.JSONDecodeError:
            pass


    return None

## 8. Model loading

Models are loaded one at a time using 4-bit quantization.

After a model finishes, it is removed from GPU memory before the next selected
model is loaded.

In [44]:
def load_model_4bit(model_id):

    print(
        f"Loading model: {model_id}"
    )


    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )


    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
    )


    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )


    model.eval()


    if tokenizer.pad_token is None:

        tokenizer.pad_token = (
            tokenizer.eos_token
        )


    print(
        f"✓ Model loaded: {model_id}"
    )


    return model, tokenizer

In [45]:
def supports_system_role(tokenizer):

    try:

        tokenizer.apply_chat_template(
            [
                {
                    "role": "system",
                    "content": "test",
                },
                {
                    "role": "user",
                    "content": "test",
                },
            ],

            tokenize=False,
            add_generation_prompt=True,
        )

        return True


    except Exception:

        return False

In [46]:
def generate_raw_response(
    model,
    tokenizer,
    context,
    question,
    max_new_tokens,
):

    user_prompt = build_user_prompt(
        context,
        question,
    )


    if supports_system_role(
        tokenizer
    ):

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },

            {
                "role": "user",
                "content": user_prompt,
            },
        ]


    else:

        combined_prompt = (
            SYSTEM_PROMPT
            + "\n\n"
            + user_prompt
        )

        messages = [
            {
                "role": "user",
                "content": combined_prompt,
            }
        ]


    input_text = (
        tokenizer
        .apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )


    inputs = tokenizer(
        input_text,
        return_tensors="pt",
    ).to(
        model.device
    )


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )


    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]


    raw_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )


    return raw_response

In [47]:
def cleanup_model(
    model,
    tokenizer,
):

    del model
    del tokenizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        "✓ GPU memory cleared."
    )

## 9. Output and checkpoint structure

Each model receives separate:

- raw output,
- metadata,
- checkpoint.

Base outputs are stored under:

```text
results/raw/base/

In [48]:
RAW_OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "raw"
    / "base"
)


CHECKPOINT_DIR = (
    RAW_OUTPUT_DIR
    / "checkpoints"
)


def safe_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("/", "-")
        .replace(" ", "-")
    )


def get_run_name(model_label):

    return (
        f"{safe_name(DATASET_PROFILE)}__"
        f"{safe_name(model_label)}__"
        f"{safe_name(RUN_LABEL)}"
    )


def get_run_paths(model_label):

    run_name = get_run_name(
        model_label
    )

    return {
        "run_name": run_name,

        "checkpoint": (
            CHECKPOINT_DIR
            / f"{run_name}.json"
        ),

        "output": (
            RAW_OUTPUT_DIR
            / f"{run_name}.json"
        ),

        "metadata": (
            RAW_OUTPUT_DIR
            / f"{run_name}.metadata.json"
        ),
    }

In [49]:
def utc_now():

    return (
        datetime
        .now(timezone.utc)
        .isoformat()
    )


def build_run_metadata(
    model_label,
):

    model_config = MODEL_REGISTRY[
        model_label
    ]


    return {
        "run_name": (
            get_run_name(
                model_label
            )
        ),

        "experiment_type": (
            "open_models_base"
        ),

        "dataset_profile": (
            DATASET_PROFILE
        ),

        "dataset_path": str(
            BASE_DATA_PATH
        ),

        "dataset_sha256": (
            DATASET_SHA256
        ),

        "raw_dataset_rows": (
            len(raw_df)
        ),

        "usable_dataset_rows": (
            len(df)
        ),

        "prompt_sha256": (
            PROMPT_SHA256
        ),

        "model_label": (
            model_label
        ),

        "model_id": (
            model_config[
                "model_id"
            ]
        ),

        "historical_model": (
            model_config[
                "historical"
            ]
        ),

        "quantization": (
            "4bit_nf4"
        ),

        "do_sample": False,

        "max_new_tokens": (
            model_config[
                "max_new_tokens"
            ]
        ),
    }

In [50]:
def save_json_atomic(
    data,
    filepath,
):

    filepath = Path(
        filepath
    )


    filepath.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_file = (
        filepath
        .with_suffix(
            filepath.suffix + ".tmp"
        )
    )


    with open(
        temporary_file,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )


    os.replace(
        temporary_file,
        filepath,
    )

In [51]:
def load_checkpoint(
    checkpoint_file,
    current_metadata,
):

    checkpoint_file = Path(
        checkpoint_file
    )


    if not checkpoint_file.exists():

        return [], None


    with open(
        checkpoint_file,
        "r",
        encoding="utf-8",
    ) as file:

        checkpoint = json.load(
            file
        )


    saved_metadata = checkpoint[
        "metadata"
    ]

    records = checkpoint[
        "records"
    ]


    fields_to_validate = [
        "dataset_sha256",
        "prompt_sha256",
        "model_id",
        "quantization",
        "max_new_tokens",
        "do_sample",
    ]


    for field in fields_to_validate:

        if (
            saved_metadata.get(field)
            != current_metadata.get(field)
        ):

            raise ValueError(
                "Checkpoint configuration mismatch "
                f"for '{field}'."
            )


    print(
        "✓ Existing checkpoint loaded: "
        f"{len(records)} records"
    )


    return (
        records,
        saved_metadata.get(
            "created_at"
        ),
    )

## 10. Generate one model over the base benchmark

Each selected model is processed independently.

The raw response is stored before evaluation.

If JSON parsing fails, the response is preserved rather than discarded.

This makes later recovery possible without rerunning the language model.

In [52]:
def run_model_generation(
    model_label,
):

    model_config = MODEL_REGISTRY[
        model_label
    ]

    model_id = model_config[
        "model_id"
    ]

    max_new_tokens = model_config[
        "max_new_tokens"
    ]


    paths = get_run_paths(
        model_label
    )

    metadata = build_run_metadata(
        model_label
    )


    records, created_at = load_checkpoint(
        paths["checkpoint"],
        metadata,
    )


    if created_at is None:

        created_at = utc_now()


    processed_ids = {
        str(
            record["question_id"]
        )

        for record in records
    }


    remaining_df = df[
        ~df["question_id"]
        .astype(str)
        .isin(
            processed_ids
        )
    ]


    print(
        f"\n{'=' * 60}"
    )

    print(
        f"MODEL: "
        f"{model_config['display_name']}"
    )

    print(
        f"{'=' * 60}"
    )

    print(
        f"Already processed: "
        f"{len(processed_ids)}"
    )

    print(
        f"Remaining: "
        f"{len(remaining_df)}"
    )


    if len(remaining_df) == 0:

        print(
            "✓ No new questions need generation."
        )

        return pd.DataFrame(
            records
        )


    model = None
    tokenizer = None


    try:

        model, tokenizer = load_model_4bit(
            model_id
        )


        for _, row in tqdm(
            remaining_df.iterrows(),
            total=len(remaining_df),
            desc=model_label,
        ):

            raw_response = None


            try:

                raw_response = generate_raw_response(
                    model=model,
                    tokenizer=tokenizer,
                    context=str(
                        row["context"]
                    ),
                    question=str(
                        row["question"]
                    ),
                    max_new_tokens=max_new_tokens,
                )


                parsed = extract_json_from_response(
                    raw_response
                )


                if parsed is None:

                    reasoning = None
                    answer = None
                    confidence = None
                    error = "JSON_PARSE_ERROR"


                else:

                    reasoning = parsed.get(
                        "reasoning"
                    )

                    answer = parsed.get(
                        "answer"
                    )

                    confidence = parsed.get(
                        "confidence"
                    )

                    error = None


            except Exception as exc:

                reasoning = None
                answer = None
                confidence = None

                error = (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )


            record = {
                "question_id": str(
                    row["question_id"]
                ),

                "category": (
                    row["category"]
                ),

                "difficulty": (
                    row["difficulty"]
                ),

                "model_label": (
                    model_label
                ),

                "model_id": (
                    model_id
                ),

                "model_reasoning": (
                    reasoning
                ),

                "model_answer": (
                    answer
                ),

                "model_confidence": (
                    confidence
                ),

                "raw_response": (
                    raw_response
                ),

                "error": (
                    error
                ),
            }


            records.append(
                record
            )


            if (
                len(records)
                % CHECKPOINT_EVERY
                == 0
            ):

                checkpoint_payload = {
                    "metadata": {
                        **metadata,

                        "created_at": (
                            created_at
                        ),

                        "updated_at": (
                            utc_now()
                        ),

                        "status": (
                            "in_progress"
                        ),
                    },

                    "records": (
                        records
                    ),
                }


                save_json_atomic(
                    checkpoint_payload,
                    paths["checkpoint"],
                )


    finally:

        if (
            model is not None
            and tokenizer is not None
        ):

            cleanup_model(
                model,
                tokenizer,
            )


    final_metadata = {
        **metadata,

        "created_at": (
            created_at
        ),

        "completed_at": (
            utc_now()
        ),

        "status": (
            "completed"
        ),

        "record_count": (
            len(records)
        ),

        "successful_records": (
            sum(
                record.get("error") is None
                for record in records
            )
        ),

        "error_records": (
            sum(
                record.get("error") is not None
                for record in records
            )
        ),
    }


    final_payload = {
        "metadata": (
            final_metadata
        ),

        "records": (
            records
        ),
    }


    save_json_atomic(
        final_payload,
        paths["output"],
    )


    save_json_atomic(
        final_metadata,
        paths["metadata"],
    )


    save_json_atomic(
        final_payload,
        paths["checkpoint"],
    )


    print(
        f"\n✓ {model_label} completed."
    )

    print(
        f"Records: {len(records)}"
    )

    print(
        f"Output: {paths['output']}"
    )


    return pd.DataFrame(
        records
    )

## 11. Generation safety gate

This is the only section that starts model execution.

While `RUN_GENERATION = False`:

- no Hugging Face login is required,
- no model is downloaded,
- no GPU generation is performed.

If multiple models are selected,
`CONFIRM_MULTIPLE_MODELS = True` must also be set explicitly.

In [53]:
generation_results = {}


if RUN_GENERATION:

    if (
        len(SELECTED_MODELS) > 1
        and not CONFIRM_MULTIPLE_MODELS
    ):

        raise RuntimeError(
            "Multiple models are selected. "
            "Set CONFIRM_MULTIPLE_MODELS = True "
            "only if you intentionally want to run them."
        )


    login_to_huggingface_if_available()


    print(
        "Generation is ENABLED."
    )

    print(
        f"Questions per model: "
        f"{len(df)}"
    )

    print(
        f"Selected models: "
        f"{len(SELECTED_MODELS)}"
    )


    for model_label in SELECTED_MODELS:

        generation_results[
            model_label
        ] = run_model_generation(
            model_label
        )


else:

    print(
        "Generation is disabled."
    )

    print(
        "No model was downloaded or executed."
    )

✓ Hugging Face authentication completed.
Generation is ENABLED.
Questions per model: 5
Selected models: 1

MODEL: Qwen 2.5 7B Instruct
Already processed: 0
Remaining: 5
Loading model: Qwen/Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✓ Model loaded: Qwen/Qwen2.5-7B-Instruct


qwen2.5-7b-instruct:   0%|          | 0/5 [00:00<?, ?it/s]

✓ GPU memory cleared.

✓ qwen2.5-7b-instruct completed.
Records: 5
Output: /content/drive/MyDrive/turkish-llm-reasoning/results/raw/base/smoke__qwen2.5-7b-instruct__smoke_v1.json


## 12. End of open-source base generation notebook

This notebook intentionally ends after raw model generation.

It does not perform:

- answer scoring,
- faithfulness evaluation,
- calibration analysis,
- model comparison,
- figure generation.

Those stages are separated so that saved responses can be reprocessed without
repeating expensive GPU generation.

The corresponding variant-generation notebook is:

`04_open_models_variants.ipynb`

All saved outputs will later be evaluated in:

`05_evaluate_saved_outputs.ipynb`